In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

from src.data.loader import BinanceLoader
from src.data.bars import create_volume_bars
from src.features.microstructure import MicrostructureEngine
from src.alpha_engine.generator import HybridAlphaEngine
from src.evaluation.walk_forward import WalkForwardEvaluator
from src.risk.portfolio import TargetVolatilityRiskManager
from src.alpha_engine.optimizer import AlphaOptimizer

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)

In [ ]:
print('Скачиваем 1m данные BTCUSDT с Binance API...')
loader = BinanceLoader()
ticks = loader.fetch_historical(symbol='BTCUSDT', start='15 days ago UTC')
ticks['price'] = ticks['close']
print(f'Загружено {len(ticks)} минут (с {ticks.index[0].date()} по {ticks.index[-1].date()})')

In [ ]:
vol_threshold = ticks['volume'].mean() * 15
bars = create_volume_bars(ticks, volume_threshold=vol_threshold)
print(f'Сформировано {len(bars)} Volume Bars.')

engine = MicrostructureEngine(window=10)
features_df = engine.fit_transform(bars)
print(f'Данные с фичами: {features_df.shape}')
display(features_df.head(3))

In [ ]:
from sklearn.preprocessing import StandardScaler
from src.koopman.edmd import EDMDKoopmanModel

feature_cols = ['log_ret', 'roll_measure', 'ofi_proxy', 'parkinson_vol']
train_w = 1000
results = []

max_start = len(features_df) - train_w - 150
if max_start <= 0:
    print('ВНИМАНИЕ: Слишком мало баров для диагностики. Уменьшите train_w или скачайте больше данных.')
else:
    for fold, start in enumerate(range(0, max_start, 150)):
        train = features_df.iloc[start: start + train_w]
        test = features_df.iloc[start + train_w: start + train_w + 150]

        scaler = StandardScaler()
        train_sc = scaler.fit_transform(train[feature_cols].values)
        test_sc = scaler.transform(test[feature_cols].values)

        model = EDMDKoopmanModel()
        model.fit(train_sc[:-1], train_sc[1:])

        preds = np.array([model.predict(test_sc[i:i + 1], steps=1).flatten()[0] for i in range(len(test_sc) - 1)])
        actual = test_sc[1:, 0]

        sign_acc = np.mean(np.sign(preds) == np.sign(actual))
        pred_mean_abs = np.mean(np.abs(preds))
        corr = np.corrcoef(preds, actual)[0, 1]

        results.append({
            'fold': fold + 1,
            'period': f"{test.index[0].strftime('%m-%d %H:%M')} → {test.index[-1].strftime('%m-%d %H:%M')}",
            'sign_acc': round(sign_acc, 3),
            'corr': round(corr, 4),
            'pred_abs': round(pred_mean_abs, 5),
        })

    diag_df = pd.DataFrame(results).set_index('fold')
    display(diag_df)
    print(f'\nСредняя точность знака : {diag_df["sign_acc"].mean():.1%}')
    print(f'Средняя корреляция     : {diag_df["corr"].mean():.4f}')

In [ ]:
param_grid = {
    'tda_window': [20, 30],
    'tda_percentile': [50.0, 70.0],
    'z_threshold': [0.0, 0.05],
    'z_quantile': [0.6, 0.7],
}

opt = AlphaOptimizer(
    param_grid,
    train_window=1000,
    test_window=150,
    verbose=True
)
results_df = opt.run(features_df)

print('\n=== ТОП-3 КОМБИНАЦИИ ===')
display(
    results_df.head(3).style
    .format({'sharpe': '{:.3f}', 'total_return': '{:.2f}%', 'max_drawdown': '{:.2f}%'})
    .background_gradient(subset=['sharpe'], cmap='RdYlGn')
)

In [ ]:
best = opt.best_params(top_n=1)[0]
print(f'Лучшие параметры: {best}')

best_engine = HybridAlphaEngine(**best)

evaluator = WalkForwardEvaluator(
    train_window=1000,
    test_window=150,
    commission=0.0005,
    slippage=0.0002
)

risk_mgr = TargetVolatilityRiskManager(target_annual_vol=0.40, max_leverage=1.0)

oos_best = evaluator.walk_forward_eval(
    features_df,
    best_engine.generate_signals,
    risk_manager=risk_mgr
)

print('\n=== МЕТРИКИ OOS ===')
print(WalkForwardEvaluator.get_summary(oos_best))

In [ ]:
fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.05)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

ax1.plot(oos_best.index, oos_best['equity_curve'], color='limegreen', lw=1.5, label='Strategy Equity')

btc_norm = oos_best['close'] / oos_best['close'].iloc[0]
ax1_r = ax1.twinx()
ax1_r.plot(oos_best.index, btc_norm, color='gray', lw=1, alpha=0.6, label='Buy & Hold BTC')
ax1_r.set_ylabel('BTC Price Growth', color='gray')

ax1.set_title('Walk-Forward OOS Equity Curve with Target Volatility Risk Manager')
ax1.set_ylabel('Cumulative Return')
ax1.legend(loc='upper left')
ax1_r.legend(loc='upper right')

ax2.bar(oos_best.index, oos_best['position'], color='steelblue', alpha=0.7, width=0.005,
        label='Position Size (-1 to 1)')
ax2.set_ylabel('Position Weight')
ax2.set_xlabel('Time')
ax2.legend(loc='upper left')

plt.setp(ax1.get_xticklabels(), visible=False)
plt.tight_layout()
plt.show()